# YawDD - Yawning Detection Model

**Yaklaşım:**
- YawDD veri setindeki her video Normal → Talking → Yawning segmentlerini sırayla içeriyor
- MediaPipe FaceLandmarker ile her frame için **Mouth Aspect Ratio (MAR)** hesaplanıyor
- MAR eşiğine göre otomatik label atılıyor: yüksek MAR = yawning, düşük MAR = normal
- Crop edilmiş ağız bölgeleriyle CNN eğitimi yapılıyor

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Gerekli kütüphaneleri yükle
!pip install mediapipe opencv-python-headless tqdm -q

In [ ]:
import os
import cv2
import numpy as np
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from tqdm import tqdm
import matplotlib.pyplot as plt
import shutil

print(f"MediaPipe version: {mp.__version__}")
print(f"OpenCV version: {cv2.__version__}")

In [ ]:
# MediaPipe FaceLandmarker modelini indir
!wget -q https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/latest/face_landmarker.task -O face_landmarker.task
print("Model indirildi.")

## 1. Veri Seti Yolu ve Video Listesi

In [ ]:
# YawDD veri seti yolu (Google Drive'daki konuma göre ayarla)
DATASET_ROOT = '/content/drive/MyDrive/havelsandataset/yawdd-dataset'

# Dash kamera videolarını kullanıyoruz (daha iyi yüz görünürlüğü)
video_dirs = [
    os.path.join(DATASET_ROOT, 'Dash', 'Dash', 'Female'),
    os.path.join(DATASET_ROOT, 'Dash', 'Dash', 'Male'),
]

# Tüm video dosyalarını topla
all_videos = []
for d in video_dirs:
    if os.path.exists(d):
        for f in sorted(os.listdir(d)):
            if f.endswith('.avi'):
                all_videos.append(os.path.join(d, f))

print(f"Toplam video sayisi: {len(all_videos)}")
for v in all_videos:
    print(os.path.basename(v))

## 2. MAR (Mouth Aspect Ratio) Hesaplama

MediaPipe FaceLandmarker'da ağız köşe ve üst/alt dudak landmarkları:
- Sol köşe: 61, Sağ köşe: 291
- Üst dudak (orta): 13
- Alt dudak (orta): 14

```
MAR = dikey_mesafe / yatay_mesafe
    = |13 - 14| / |61 - 291|
```

Yüksek MAR (> eşik) → ağız açık → yawning

In [ ]:
# MediaPipe FaceLandmarker'ı başlat
BaseOptions = mp.tasks.BaseOptions
FaceLandmarker = mp.tasks.vision.FaceLandmarker
FaceLandmarkerOptions = mp.tasks.vision.FaceLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

options = FaceLandmarkerOptions(
    base_options=BaseOptions(model_asset_path='face_landmarker.task'),
    running_mode=VisionRunningMode.IMAGE,
    num_faces=1,
    min_face_detection_confidence=0.5,
    min_face_presence_confidence=0.5,
    min_tracking_confidence=0.5
)

# Ağız landmark indeksleri (MediaPipe 478 nokta modeli)
MOUTH_LEFT   = 61
MOUTH_RIGHT  = 291
MOUTH_TOP    = 13
MOUTH_BOTTOM = 14

# Ağız bölgesi crop için kullanılan tüm dış sınır noktaları
MOUTH_OUTLINE = [61, 185, 40, 39, 37, 0, 267, 269, 270, 409, 291,
                 375, 321, 405, 314, 17, 84, 181, 91, 146]

def compute_mar(landmarks, img_w, img_h):
    """Mouth Aspect Ratio hesapla"""
    lm = landmarks
    left  = np.array([lm[MOUTH_LEFT].x * img_w,  lm[MOUTH_LEFT].y * img_h])
    right = np.array([lm[MOUTH_RIGHT].x * img_w, lm[MOUTH_RIGHT].y * img_h])
    top   = np.array([lm[MOUTH_TOP].x * img_w,   lm[MOUTH_TOP].y * img_h])
    bottom= np.array([lm[MOUTH_BOTTOM].x * img_w,lm[MOUTH_BOTTOM].y * img_h])

    horizontal = np.linalg.norm(right - left)
    vertical   = np.linalg.norm(bottom - top)

    if horizontal < 1e-6:
        return 0.0
    return float(vertical / horizontal)

def crop_mouth(frame, landmarks, img_w, img_h, padding=0.3):
    """Ağız bölgesini crop et"""
    lm = landmarks
    xs = [lm[i].x * img_w for i in MOUTH_OUTLINE]
    ys = [lm[i].y * img_h for i in MOUTH_OUTLINE]

    x_min, x_max = min(xs), max(xs)
    y_min, y_max = min(ys), max(ys)

    w = x_max - x_min
    h = y_max - y_min

    pad_x = w * padding
    pad_y = h * padding

    x1 = max(0, int(x_min - pad_x))
    y1 = max(0, int(y_min - pad_y))
    x2 = min(img_w, int(x_max + pad_x))
    y2 = min(img_h, int(y_max + pad_y))

    crop = frame[y1:y2, x1:x2]
    return crop

print("Fonksiyonlar hazir.")

## 3. MAR Dağılımını Analiz Et (Eşik Belirleme)

Önce birkaç videodan MAR değerlerini çıkarıp dağılımına bakıyoruz.

In [ ]:
# Birkaç videodan MAR değerlerini analiz et
SAMPLE_VIDEOS = all_videos[:5]  # İlk 5 videoyu analiz et
FRAME_SKIP = 5  # Her 5 frame'de bir işle (hız için)

mar_values_all = []

with FaceLandmarker.create_from_options(options) as detector:
    for video_path in SAMPLE_VIDEOS:
        cap = cv2.VideoCapture(video_path)
        frame_idx = 0
        video_name = os.path.basename(video_path)
        mar_values_video = []

        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
            frame_idx += 1
            if frame_idx % FRAME_SKIP != 0:
                continue

            h, w = frame.shape[:2]
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)

            result = detector.detect(mp_image)
            if result.face_landmarks:
                mar = compute_mar(result.face_landmarks[0], w, h)
                mar_values_video.append(mar)

        cap.release()
        mar_values_all.extend(mar_values_video)
        print(f"{video_name}: {len(mar_values_video)} frame islenedi, "
              f"MAR ort={np.mean(mar_values_video):.3f}, max={np.max(mar_values_video):.3f}")

print(f"\nToplam frame: {len(mar_values_all)}")

In [ ]:
# MAR dağılımını görselleştir
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.hist(mar_values_all, bins=80, color='steelblue', edgecolor='white')
plt.xlabel('MAR Degeri')
plt.ylabel('Frame Sayisi')
plt.title('MAR Dagilimi')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.boxplot(mar_values_all)
plt.ylabel('MAR Degeri')
plt.title('MAR Boxplot')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Yüzdelikler
for p in [50, 70, 80, 85, 90, 95]:
    print(f"%{p} = {np.percentile(mar_values_all, p):.4f}")

In [ ]:
# MAR eşiğini belirle
# Genellikle üst %15-20 yawning olarak kabul edilir
# Dağılıma göre bu değeri ayarla

MAR_THRESHOLD = np.percentile(mar_values_all, 82)  # Üst %18 = yawning
MAR_NORMAL_MAX = np.percentile(mar_values_all, 40)  # Alt %40 = normal (belirsiz orta bölgeyi atla)

print(f"MAR Esigi (yawning)  : {MAR_THRESHOLD:.4f}")
print(f"MAR Esigi (normal)   : {MAR_NORMAL_MAX:.4f}")
print(f"Orta bolge atlanacak: {MAR_NORMAL_MAX:.4f} - {MAR_THRESHOLD:.4f}")

# Görsel olarak göster
plt.figure(figsize=(10, 4))
plt.hist(mar_values_all, bins=80, color='steelblue', edgecolor='white', label='Tum frameler')
plt.axvline(MAR_THRESHOLD, color='red', linewidth=2, label=f'Yawning esigi ({MAR_THRESHOLD:.3f})')
plt.axvline(MAR_NORMAL_MAX, color='green', linewidth=2, label=f'Normal esigi ({MAR_NORMAL_MAX:.3f})')
plt.xlabel('MAR')
plt.ylabel('Frame Sayisi')
plt.title('MAR Esik Degerleri')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 4. Tüm Videolardan Frame Çıkar ve Kaydet

In [ ]:
# Çıktı dizinleri
OUTPUT_DIR = '/content/yawning_dataset'
YAWNING_DIR = os.path.join(OUTPUT_DIR, 'yawning')
NORMAL_DIR  = os.path.join(OUTPUT_DIR, 'normal')

os.makedirs(YAWNING_DIR, exist_ok=True)
os.makedirs(NORMAL_DIR,  exist_ok=True)

IMG_SIZE = 64  # Crop boyutu
FRAME_SKIP = 3  # Her 3 frame'de bir kaydet
MAX_PER_CLASS_PER_VIDEO = 150  # Sınıf dengesi için video başına max frame

yawning_count = 0
normal_count  = 0
skipped_no_face = 0

with FaceLandmarker.create_from_options(options) as detector:
    for video_path in tqdm(all_videos, desc='Videolar isleniyor'):
        cap = cv2.VideoCapture(video_path)
        video_name = os.path.splitext(os.path.basename(video_path))[0]
        frame_idx = 0
        video_yawning = 0
        video_normal  = 0

        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
            frame_idx += 1
            if frame_idx % FRAME_SKIP != 0:
                continue

            # Video başına max frame kontrolü
            if video_yawning >= MAX_PER_CLASS_PER_VIDEO and video_normal >= MAX_PER_CLASS_PER_VIDEO:
                break

            h, w = frame.shape[:2]
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)

            result = detector.detect(mp_image)
            if not result.face_landmarks:
                skipped_no_face += 1
                continue

            mar = compute_mar(result.face_landmarks[0], w, h)
            mouth_crop = crop_mouth(frame, result.face_landmarks[0], w, h)

            if mouth_crop.size == 0:
                continue

            mouth_resized = cv2.resize(mouth_crop, (IMG_SIZE, IMG_SIZE))

            if mar >= MAR_THRESHOLD and video_yawning < MAX_PER_CLASS_PER_VIDEO:
                fname = f"{video_name}_f{frame_idx:05d}.jpg"
                cv2.imwrite(os.path.join(YAWNING_DIR, fname), mouth_resized)
                yawning_count += 1
                video_yawning += 1

            elif mar <= MAR_NORMAL_MAX and video_normal < MAX_PER_CLASS_PER_VIDEO:
                fname = f"{video_name}_f{frame_idx:05d}.jpg"
                cv2.imwrite(os.path.join(NORMAL_DIR, fname), mouth_resized)
                normal_count += 1
                video_normal += 1

        cap.release()

print(f"\nFrame cikartma tamamlandi!")
print(f"  Yawning  : {yawning_count} frame")
print(f"  Normal   : {normal_count} frame")
print(f"  Yuz yok  : {skipped_no_face} frame atildi")

In [ ]:
# Örnek görsellere bak
fig, axes = plt.subplots(2, 8, figsize=(16, 5))

for row, (label, folder) in enumerate([("Yawning", YAWNING_DIR), ("Normal", NORMAL_DIR)]):
    files = sorted(os.listdir(folder))[:8]
    for col, fname in enumerate(files):
        img = cv2.imread(os.path.join(folder, fname))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        axes[row, col].imshow(img)
        axes[row, col].axis('off')
        if col == 0:
            axes[row, col].set_title(label, fontsize=12, fontweight='bold')

plt.suptitle('Ornek Crop Gorseller', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 5. Train/Val/Test Split

In [ ]:
import random
from sklearn.model_selection import train_test_split

SPLIT_DIR = '/content/yawning_split'

for split in ['train', 'val', 'test']:
    for cls in ['yawning', 'normal']:
        os.makedirs(os.path.join(SPLIT_DIR, split, cls), exist_ok=True)

def split_and_copy(src_dir, class_name, train_r=0.7, val_r=0.15):
    files = [f for f in os.listdir(src_dir) if f.endswith('.jpg')]
    random.shuffle(files)

    n = len(files)
    n_train = int(n * train_r)
    n_val   = int(n * val_r)

    splits = {
        'train': files[:n_train],
        'val'  : files[n_train:n_train + n_val],
        'test' : files[n_train + n_val:]
    }

    for split_name, split_files in splits.items():
        dst = os.path.join(SPLIT_DIR, split_name, class_name)
        for f in split_files:
            shutil.copy(os.path.join(src_dir, f), os.path.join(dst, f))
        print(f"  {split_name}/{class_name}: {len(split_files)} gorsel")

print("Yawning split:")
split_and_copy(YAWNING_DIR, 'yawning')
print("Normal split:")
split_and_copy(NORMAL_DIR, 'normal')

## 6. CNN Model Eğitimi

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator

print(f"TensorFlow: {tf.__version__}")
print(f"GPU mevcut: {len(tf.config.list_physical_devices('GPU')) > 0}")

IMG_HEIGHT = 64
IMG_WIDTH  = 64
BATCH_SIZE = 64
EPOCHS     = 50

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2],
    zoom_range=0.1
)

val_test_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory(
    os.path.join(SPLIT_DIR, 'train'),
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=True
)

val_gen = val_test_datagen.flow_from_directory(
    os.path.join(SPLIT_DIR, 'val'),
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False
)

test_gen = val_test_datagen.flow_from_directory(
    os.path.join(SPLIT_DIR, 'test'),
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False
)

class_names = list(train_gen.class_indices.keys())
print(f"\nSiniflar: {train_gen.class_indices}")

In [ ]:
# CNN Modeli
model = keras.Sequential([
    # Blok 1
    layers.Conv2D(32, (3,3), activation='relu', padding='same',
                  input_shape=(IMG_HEIGHT, IMG_WIDTH, 3)),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),
    layers.Dropout(0.25),

    # Blok 2
    layers.Conv2D(64, (3,3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),
    layers.Dropout(0.25),

    # Blok 3
    layers.Conv2D(128, (3,3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),
    layers.Dropout(0.25),

    # Blok 4
    layers.Conv2D(256, (3,3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.4),

    # Dense
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(1, activation='sigmoid')
])

model.summary()

In [ ]:
from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy('mixed_float16')

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

callbacks = [
    keras.callbacks.ModelCheckpoint(
        'best_yawning_model.h5',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    ),
    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=7,
        restore_best_weights=True,
        verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-7,
        verbose=1
    )
]

print("Model derlendi.")

In [ ]:
print("Egitim basliyor...\n")

history = model.fit(
    train_gen,
    epochs=EPOCHS,
    validation_data=val_gen,
    callbacks=callbacks,
    verbose=1
)

print("\nEgitim tamamlandi!")

## 7. Eğitim Sonuçları

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history.history['accuracy'],     label='Train Accuracy', linewidth=2)
ax1.plot(history.history['val_accuracy'], label='Val Accuracy',   linewidth=2)
ax1.set_title('Model Accuracy')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(history.history['loss'],     label='Train Loss', linewidth=2)
ax2.plot(history.history['val_loss'], label='Val Loss',   linewidth=2)
ax2.set_title('Model Loss')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"En iyi Train Accuracy : {max(history.history['accuracy']):.4f}")
print(f"En iyi Val Accuracy   : {max(history.history['val_accuracy']):.4f}")

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

best_model = keras.models.load_model('best_yawning_model.h5')

# Test değerlendirmesi
test_loss, test_acc = best_model.evaluate(test_gen, verbose=0)
print(f"Test Accuracy : {test_acc*100:.2f}%")
print(f"Test Loss     : {test_loss:.4f}")

# Confusion matrix
test_gen.reset()
preds = best_model.predict(test_gen, verbose=0)
pred_classes = (preds > 0.5).astype(int).flatten()
true_classes = test_gen.classes

print("\nClassification Report:")
print(classification_report(true_classes, pred_classes, target_names=class_names))

cm = confusion_matrix(true_classes, pred_classes)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix')
plt.ylabel('Gercek')
plt.xlabel('Tahmin')
plt.tight_layout()
plt.show()

## 8. Modeli Kaydet

In [ ]:
# Modeli Google Drive'a kaydet
save_path = '/content/drive/MyDrive/havelsandataset/yawning_model.h5'
best_model.save(save_path)
print(f"Model kaydedildi: {save_path}")

# MAR eşik değerlerini de kaydet (inference için gerekli)
import json
thresholds = {
    'mar_yawning_threshold': float(MAR_THRESHOLD),
    'mar_normal_max': float(MAR_NORMAL_MAX),
    'img_size': IMG_SIZE,
    'class_indices': train_gen.class_indices
}
thresh_path = '/content/drive/MyDrive/havelsandataset/yawning_thresholds.json'
with open(thresh_path, 'w') as f:
    json.dump(thresholds, f, indent=2)
print(f"Esik degerleri kaydedildi: {thresh_path}")
print(json.dumps(thresholds, indent=2))

## 9. Tek Frame Üzerinde Test (Inference Demo)

In [ ]:
def predict_yawning(frame_bgr, detector, model, img_size=64):
    """
    Tek bir frame üzerinde yawning tahmini yap.
    Returns: (label, confidence, mar)
    """
    h, w = frame_bgr.shape[:2]
    rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)

    result = detector.detect(mp_image)
    if not result.face_landmarks:
        return None, 0.0, 0.0

    mar = compute_mar(result.face_landmarks[0], w, h)
    mouth = crop_mouth(frame_bgr, result.face_landmarks[0], w, h)

    if mouth.size == 0:
        return None, 0.0, mar

    mouth_resized = cv2.resize(mouth, (img_size, img_size))
    mouth_norm = mouth_resized.astype(np.float32) / 255.0
    mouth_input = np.expand_dims(mouth_norm, axis=0)  # (1, 64, 64, 3)

    pred = model.predict(mouth_input, verbose=0)[0][0]

    # class_indices: {'normal': 0, 'yawning': 1} veya tersi olabilir
    # yawning sınıfının indeksine göre ayarla
    yawning_idx = train_gen.class_indices.get('yawning', 1)
    if yawning_idx == 1:
        yawning_prob = pred
    else:
        yawning_prob = 1.0 - pred

    label = 'yawning' if yawning_prob > 0.5 else 'normal'
    return label, float(yawning_prob), float(mar)

# Örnek test
sample_video = all_videos[0]
cap = cv2.VideoCapture(sample_video)

with FaceLandmarker.create_from_options(options) as detector:
    results_demo = []
    for _ in range(200):
        ret, frame = cap.read()
        if not ret:
            break
        label, conf, mar = predict_yawning(frame, detector, best_model)
        if label:
            results_demo.append((label, conf, mar))

cap.release()

yawning_frames = sum(1 for r in results_demo if r[0] == 'yawning')
print(f"Demo video ({os.path.basename(sample_video)}):")
print(f"  Toplam frame   : {len(results_demo)}")
print(f"  Yawning frame  : {yawning_frames} ({yawning_frames/len(results_demo)*100:.1f}%)")
print(f"  Normal frame   : {len(results_demo) - yawning_frames}")